In [207]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [208]:
data_transformer = transforms.ToTensor()

In [209]:
embedding_size = 128
to_decoder_dim = 64
mask_ratio = 0.75
num_attn_heads = 4
num_of_hidden_nodes = 2048
num_of_encoder_blocks = 8
num_of_decoder_blocks = 4
input_size = 32
patch_size = 2
batch_size = 64

In [210]:
train_dataset = datasets.CIFAR10(
    root = './data',
    train = True,
    download = True, 
    transform = data_transformer

)

val_dataset = datasets.CIFAR10(
    root = './data',
    train = False,
    download = True, 
    transform = data_transformer

)

In [211]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)

In [212]:
print(train_dataset.classes)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [213]:
images, labels = next(iter(train_loader))
print(images.shape)

torch.Size([64, 3, 32, 32])


In [214]:
images[0][1][3]

tensor([0.4118, 0.4314, 0.4980, 0.5961, 0.6588, 0.6902, 0.7059, 0.7059, 0.7176,
        0.7255, 0.7255, 0.7216, 0.7412, 0.7608, 0.7451, 0.7490, 0.8196, 0.8667,
        0.8824, 0.8941, 0.8902, 0.8745, 0.8275, 0.7686, 0.7961, 0.7569, 0.6627,
        0.6078, 0.5843, 0.5647, 0.5451, 0.5412])

In [215]:
#converting the images into patches
patches = F.unfold(images, kernel_size=2, stride=2)
print("before transpose",patches.shape) #before transpose
patches = patches.transpose(1,2)
print("before transpose",patches.shape) #after transpose
#before patch creation - (64,3,32,32)
#Output (after transpose) - (64,256,12) 64 - batch, 256 patches (as 32/2 = 16 --> 16x16=256) , 12 flattened vector got by - 2x2-->4x3 = 12
# all channel values are flattened 

before transpose torch.Size([64, 12, 256])
before transpose torch.Size([64, 256, 12])


In [216]:
#embedding the patches to increase information representation
embedder = nn.Linear(12, embedding_size)
patches = embedder(patches)
print(patches.shape)
# (B, P, C)
#B- batches, P - Patches,  C - dim/channels (got flatten up)

torch.Size([64, 256, 128])


In [217]:
#defining learnable position embeddings and adding them
pos_embed = nn.Parameter(torch.randn(1,256,128))
patches = patches + pos_embed
print(patches.shape)

torch.Size([64, 256, 128])


In [218]:
num_patches = patches.shape[1]

In [219]:
perm_nums = torch.randperm(num_patches)
print(perm_nums)

tensor([  0, 202,  81, 142,   5, 215, 177, 228, 178, 172, 226,  22, 237, 189,
        115,  33,   6,  51, 128, 129, 112, 159,  40, 186, 196,  88, 234,  73,
        139, 161, 240, 190,  14,  86, 169,  83,  60, 230, 150,  41, 176,  68,
         24, 108, 248, 111, 132, 135,  46, 166,   2,  80, 241, 199,  29, 204,
         66, 208, 123, 255,  58, 192, 155,  34,   1, 119,  61, 121, 127,  38,
        242,  94, 118,  91, 213, 116, 180, 183, 200, 145, 205, 141, 250, 236,
         45, 243, 152,  28,  30, 124, 156, 229, 198, 137, 245, 254, 171, 217,
         39, 151, 210,  62, 224, 209, 194,  16,  50,  43,  85, 146,  12, 225,
        182, 223, 221, 102, 148, 227, 110, 147, 140,  90, 138, 216, 104,  89,
         78,  47, 203, 173,  67,  49, 253,  37, 195, 188, 125, 133,  26, 105,
         99, 219,  76, 167, 109, 249, 144, 218, 134,  44, 143,  97, 207, 153,
         96, 165, 136,  95, 244,   3, 170, 247,  53, 163, 130, 246,  18,   9,
          8,  55,  48, 239, 212, 164, 114,  27, 106,  11,  69,  

In [220]:
mask_idx_count = int(num_patches*mask_ratio)
visible_idx_count = num_patches - mask_idx_count
print("patches selected for masking : ", mask_idx_count)
print("Patches unselected from masking : ", visible_idx_count)

patches selected for masking :  192
Patches unselected from masking :  64


In [221]:
mask_idxs = perm_nums[:mask_idx_count]
visible_idxs = perm_nums[mask_idx_count:]

print(mask_idxs)
print(visible_idxs)

tensor([  0, 202,  81, 142,   5, 215, 177, 228, 178, 172, 226,  22, 237, 189,
        115,  33,   6,  51, 128, 129, 112, 159,  40, 186, 196,  88, 234,  73,
        139, 161, 240, 190,  14,  86, 169,  83,  60, 230, 150,  41, 176,  68,
         24, 108, 248, 111, 132, 135,  46, 166,   2,  80, 241, 199,  29, 204,
         66, 208, 123, 255,  58, 192, 155,  34,   1, 119,  61, 121, 127,  38,
        242,  94, 118,  91, 213, 116, 180, 183, 200, 145, 205, 141, 250, 236,
         45, 243, 152,  28,  30, 124, 156, 229, 198, 137, 245, 254, 171, 217,
         39, 151, 210,  62, 224, 209, 194,  16,  50,  43,  85, 146,  12, 225,
        182, 223, 221, 102, 148, 227, 110, 147, 140,  90, 138, 216, 104,  89,
         78,  47, 203, 173,  67,  49, 253,  37, 195, 188, 125, 133,  26, 105,
         99, 219,  76, 167, 109, 249, 144, 218, 134,  44, 143,  97, 207, 153,
         96, 165, 136,  95, 244,   3, 170, 247,  53, 163, 130, 246,  18,   9,
          8,  55,  48, 239, 212, 164, 114,  27, 106,  11,  69,  

In [222]:
masked_patches = patches[:, mask_idxs , :]
visible_patches = patches[:, visible_idxs, :]

print(masked_patches.shape)
print(visible_patches.shape)

torch.Size([64, 192, 128])
torch.Size([64, 64, 128])


In [223]:
class Patchify(nn.Module):
    def __init__(self):
        super().__init__()
     
        

    def forward(self,x):
        patches = F.unfold(x, kernel_size=2, stride=2)
        patches = patches.transpose(-1,-2)
        return patches


In [224]:
class Embed_patches_add_pos(nn.Module):
    def __init__(self, embedding_size, no_of_patches):
        super().__init__()
        self.embedding_dim = embedding_size
     
        self.no_of_patches = no_of_patches
        self.embedder = nn.Linear(12, self.embedding_dim)
        self.pos_embed = nn.Parameter(torch.randn(1,self.no_of_patches,self.embedding_dim))


    def forward(self,x):
        patchx = self.embedder(x)
        patchx = patchx + self.pos_embed

        return patchx



In [225]:
class Mask(nn.Module):
    def __init__(self, num_patches, mask_ratio):
        super().__init__()
        self.num_patches = num_patches
        self.mask_ratio = mask_ratio
        self.perm_nums = torch.randperm(num_patches)

    def forward(self):
        mask_idx_count = int(self.num_patches*self.mask_ratio)
        visible_idx_count = self.num_patches - mask_idx_count
        mask_idxs = self.perm_nums[:mask_idx_count]
        visible_idxs = self.perm_nums[mask_idx_count:]
        masked_patches = patches[:, mask_idxs , :]
        visible_patches = patches[:, visible_idxs, :]

        return visible_patches, mask_idxs, self.perm_nums
        



In [226]:
#MultiHeadAttention
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim,num_patches,num_heads  ):
        super().__init__()
        #definitions
        #input and #output projections
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.num_patches = num_patches
        self.dim_per_head = self.embedding_dim//self.num_heads

        self.q = nn.Linear(self.embedding_dim, self.embedding_dim)
        self.k = nn.Linear(self.embedding_dim, self.embedding_dim)
        self.v = nn.Linear(self.embedding_dim, self.embedding_dim)
        
        self.proj = nn.Linear(self.embedding_dim, self.embedding_dim)

        #we need to divide the dotproduct with sqroot(dim_per_head) as scalars of dot product
        #increase as vector increases, and that leads to softmax being too peaky, meaning
        #softmax gives maximum attention/value to the bigger values so eventually smaller values are not given much of importance
        #so occurs vanishing gradient problem
        #variance of dot-product grow proportionally to dimension, 
        self.scale =self.dim_per_head**(-0.5)

    def split_heads(self, x):
        batch_sizee , seq_len, emb_dim = x.shape
        x = x.view(batch_sizee, seq_len, self.num_heads, self.dim_per_head )
        return x.permute(0,2,1,3)
    

    def merge_heads(self,x):  
        x = x.permute(0,2,1,3).contiguous()
        batch_sizee , seq_len, num_headss, dim_per_headd = x.shape
        return x.view(batch_sizee, seq_len, self.embedding_dim)

    def forward(self,x):
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        #K = (64,4,256,32)
        #Q = (64,4,256,32)
        #K.T = (64, 4,32,256)
        #Q@K.T = (64,4, 256, 256)
        self.wei = Q@K.transpose(-1,-2)
        self.wei = self.wei * self.scale
        attn = F.softmax(self.wei, dim = -1)
        self.attn_map = attn
        self.output = attn@V

        self.output = self.merge_heads(self.output)
        return self.proj(self.output)

In [227]:
class MLP(nn.Module):
    def __init__(self, embedding_dim, num_hidden_nodes, dropout=0.1):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_hidden_nodes = num_hidden_nodes
        self.fc1 = nn.Linear(self.embedding_dim, self.num_hidden_nodes)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(self.num_hidden_nodes, self.embedding_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x


In [228]:
class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim , num_patches, num_attn_heads, num_of_hidden_nodes):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_patches = num_patches
        self.num_attn_heads = num_attn_heads
        self.num_hidden_nodes = num_of_hidden_nodes

        self.norm1 = nn.LayerNorm(self.embedding_dim)
        self.norm2 = nn.LayerNorm(self.embedding_dim)

        self.attention = MultiHeadAttention(self.embedding_dim, self.num_patches, self.num_attn_heads)
        self.mlp = MLP(self.embedding_dim, self.num_hidden_nodes, dropout=0.25)

    def forward(self, x):
        residual1 = x
        x = self.norm1(x)
        attn_opt = self.attention(x)
        attn_opt = attn_opt + residual1

        residual2 = attn_opt
        x = self.norm2(attn_opt)
        mlp_opt = self.mlp(x)
        mlp_opt = mlp_opt+residual2

        return mlp_opt


In [229]:
class DecoderBlock(nn.Module):
    def __init__(self, embedding_dim , num_patches, num_attn_heads, num_of_hidden_nodes):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_patches = num_patches
        self.num_attn_heads = num_attn_heads
        self.num_hidden_nodes = num_of_hidden_nodes

        self.norm1 = nn.LayerNorm(self.embedding_dim)
        self.norm2 = nn.LayerNorm(self.embedding_dim)

        self.attention = MultiHeadAttention(self.embedding_dim, self.num_patches, self.num_attn_heads)
        self.mlp = MLP(self.embedding_dim, self.num_hidden_nodes, dropout=0.25)

    def forward(self, x):
        residual1 = x
        x = self.norm1(x)
        attn_opt = self.attention(x)
        attn_opt = attn_opt + residual1

        residual2 = attn_opt
        x = self.norm2(attn_opt)
        mlp_opt = self.mlp(x)
        mlp_opt = mlp_opt+residual2

        return mlp_opt
        

In [230]:
class MAE(nn.Module):
    def __init__(self, batch_size,input_size, patch_size,mask_ratio, embedding_dim , num_attn_heads, num_of_hidden_nodes,num_of_encoder_blocks,num_of_decoder_blocks,to_decoder_dim ):
        super().__init__()
        
        self.batch_size = batch_size
        self.input_size = input_size
        self.patch_size = patch_size
        self.embedding_dim = embedding_dim
        self.num_patches = (self.input_size // self.patch_size)**2
        self.num_attn_heads = num_attn_heads
        self.num_of_hidden_nodes = num_of_hidden_nodes
        self.num_of_encoder_blocks = num_of_encoder_blocks
        self.num_of_decoder_blocks = num_of_decoder_blocks
        self.to_decoder_dim = to_decoder_dim
        self.mask_ratio = mask_ratio
        self.num_of_masked = int(self.num_patches*self.mask_ratio)

        self.layer_to_dec_dim = nn.Linear(self.embedding_dim, self.to_decoder_dim)
        self.mask_tokens = nn.Parameter(torch.randn(1, self.num_of_masked, self.to_decoder_dim))
        self.to_original_dim = nn.Linear(self.to_decoder_dim, 12)

        self.patching = Patchify()
        self.emb_posemb = Embed_patches_add_pos(self.embedding_dim,self.num_patches)
        self.masking = Mask(self.num_patches, self.mask_ratio)

        self.encoder = EncoderBlock(self.embedding_dim, self.num_patches, self.num_attn_heads, self.num_of_hidden_nodes)
        self.decoder = DecoderBlock(self.embedding_dim, self.num_patches, self.num_attn_heads, self.num_of_hidden_nodes)



    def forward(self,x):
        patches = self.patching(x)
        patches = self.emb_posemb(patches)
        visible_patches , mask_idxs , perm_nums = self.masking(patches)

        latent = self.encoder(visible_patches)
        latent = self.layer_to_dec_dim(latent)
        all_tokens =  torch.cat([latent, self.mask_tokens], dim=1)
        restored = all_tokens[:,torch.argsort(perm_nums), :]


        decoder_opt = self.decoder(restored)
        pred_pixels = self.to_original_dim(decoder_opt)

        reconstructed = F.fold(pred_pixels, (32,32), kernel_size=2, stride=2)

        pred_masked = pred_pixels[:, mask_idxs, :]
        target_masked = patches[:, mask_idxs, :]

        loss = F.mse_loss(pred_masked, target_masked)

        return reconstructed, loss





        

        


In [231]:
model = MAE(
    batch_size=batch_size,
    embedding_dim=embedding_size,
    patch_size=patch_size,
    mask_ratio=mask_ratio,
    input_size=input_size,
    num_attn_heads=num_attn_heads,
    num_of_hidden_nodes=num_of_hidden_nodes,
    num_of_encoder_blocks=num_of_encoder_blocks,
    num_of_decoder_blocks=num_of_decoder_blocks,
    to_decoder_dim=to_decoder_dim
)

In [232]:
criterion = nn.CrossEntropyLoss()

In [233]:
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-5)


In [234]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [235]:
model = model.to(device=device)

In [236]:
from torchinfo import summary
print(summary(model))

Layer (type:depth-idx)                   Param #
MAE                                      12,288
├─Linear: 1-1                            8,256
├─Linear: 1-2                            780
├─Patchify: 1-3                          --
├─Embed_patches_add_pos: 1-4             32,768
│    └─Linear: 2-1                       1,664
├─Mask: 1-5                              --
├─EncoderBlock: 1-6                      --
│    └─LayerNorm: 2-2                    256
│    └─LayerNorm: 2-3                    256
│    └─MultiHeadAttention: 2-4           --
│    │    └─Linear: 3-1                  16,512
│    │    └─Linear: 3-2                  16,512
│    │    └─Linear: 3-3                  16,512
│    │    └─Linear: 3-4                  16,512
│    └─MLP: 2-5                          --
│    │    └─Linear: 3-5                  264,192
│    │    └─GELU: 3-6                    --
│    │    └─Linear: 3-7                  262,272
│    │    └─Dropout: 3-8                 --
├─DecoderBlock: 1-7         

In [237]:
epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    total_re_loss = 0
    combined_total_loss = 0

    for images in train_loader:
        images = images.to(device)
        output , re_loss = model(images)

        loss = criterion(output, images)
        optimizer.zero_grad()
        loss = loss + re_loss
        loss.backward()
        optimizer.step()
        total_loss +=loss.item()
        total_re_loss+= loss.item()
        combined_total_loss+= total_re_loss+total_loss

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

AttributeError: 'list' object has no attribute 'to'